In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor


# 학습 모델 저장을 위한 라이브러리
import pickle

### 프로젝트 셋팅

In [2]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_final14.dat'
# 교차검증 횟수
cv_count = 10
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

### 데이터 준비
- 데이터를 읽어오고 필요한 전처리까지 다 한다음 입력데이터는 train_X, 결과데이터는 train_y라는 변수에 담아서 준비해주세요

In [3]:
# 데이터를 읽어온다.
train_df = pd.read_parquet('data/Seleted(A_B_notAB)_delecteE_all_train.parquet')
test_df = pd.read_parquet('data/Seleted(A_B_notAB)_delecteE_all_test.parquet')

display(train_df)
display(test_df)

,기준년월,ID,Group,카드이용한도금액_B2M,카드이용한도금액_B1M,카드이용한도금액,CA한도금액,_1순위카드이용금액,연령,Life_Stage,...,청구금액_R3M,청구금액_B0,포인트_마일리지_환산_B0M,마일_적립포인트_R3M,할인건수_R3M,평잔_일시불_해외_6M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M
0,201807,TRAIN_000000,notAB,19723,20805,19354,7270,3681,40대,자녀성장(2),...,46588,12226,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,10회 이상
1,201807,TRAIN_000002,notAB,77975,78730,88193,35207,24493,30대,자녀출산기,...,85931,21866,0,0,1회 이상,0,30회 이상,10회 이상,10회 이상,1회 이상
2,201807,TRAIN_000003,notAB,19226,20523,19062,6531,5933,40대,자녀성장(2),...,61518,16356,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,10회 이상
3,201807,TRAIN_000008,notAB,208224,199966,199862,62007,68078,30대,자녀출산기,...,62715,20512,0,0,10회 이상,559,1회 이상,1회 이상,1회 이상,1회 이상
4,201807,TRAIN_000010,notAB,100056,100037,100017,31378,18796,40대,자녀성장(1),...,30449,22512,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
477943,201812,TRAIN_399979,notAB,84977,76798,88823,26268,27337,40대,자녀성장(2),...,41812,11817,0,0,1회 이상,365,1회 이상,1회 이상,1회 이상,1회 이상
477944,201812,TRAIN_399987,notAB,37740,37302,42192,10026,35751,50대,자녀성장(2),...,68356,17859,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,10회 이상
477945,201812,TRAIN_399993,notAB,61134,60057,53999,22513,27792,40대,자녀성장(1),...,34890,10810,0,0,20회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
477946,201812,TRAIN_399996,notAB,78140,78997,84217,31159,26357,50대,자녀성장(2),...,37515,14402,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상


,기준년월,ID,카드이용한도금액_B2M,카드이용한도금액_B1M,카드이용한도금액,CA한도금액,_1순위카드이용금액,연령,Life_Stage,거주시도명,...,청구금액_R3M,청구금액_B0,포인트_마일리지_환산_B0M,마일_적립포인트_R3M,할인건수_R3M,평잔_일시불_해외_6M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M
0,201807,TEST_00000,50006,49999,50902,18131,13852,40대,자녀성장(1),경기,...,11441,4931,148,2532,1회 이상,384,1회 이상,1회 이상,1회 이상,1회 이상
1,201807,TEST_00001,50003,50000,50080,16819,11065,60대,자녀독립기,인천,...,20522,10152,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
2,201807,TEST_00002,100056,100053,100045,30505,27071,40대,자녀성장(1),경기,...,50508,13223,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
3,201807,TEST_00003,19693,21035,18508,6402,4827,40대,자녀성장(1),인천,...,4604,2112,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
4,201807,TEST_00004,3924,4291,4033,0,8011,40대,자녀성장(1),경기,...,6788,4406,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,201812,TEST_99995,0,0,0,0,0,60대,노년생활,경기,...,0,0,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
599996,201812,TEST_99996,49990,50008,49025,17876,1231,30대,자녀출산기,서울,...,1256,359,0,0,1회 이상,0,10회 이상,1회 이상,1회 이상,1회 이상
599997,201812,TEST_99997,30004,30011,29996,13332,0,30대,자녀성장(1),경남,...,0,0,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
599998,201812,TEST_99998,37139,37999,42610,17362,63592,30대,가족구축기,경남,...,48141,21273,0,0,1회 이상,240,40회 이상,1회 이상,1회 이상,1회 이상


In [4]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df])
all_df.reset_index(inplace=True, drop=True)
all_df

,기준년월,ID,Group,카드이용한도금액_B2M,카드이용한도금액_B1M,카드이용한도금액,CA한도금액,_1순위카드이용금액,연령,Life_Stage,...,청구금액_R3M,청구금액_B0,포인트_마일리지_환산_B0M,마일_적립포인트_R3M,할인건수_R3M,평잔_일시불_해외_6M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M
0,201807,TRAIN_000000,notAB,19723,20805,19354,7270,3681,40대,자녀성장(2),...,46588,12226,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,10회 이상
1,201807,TRAIN_000002,notAB,77975,78730,88193,35207,24493,30대,자녀출산기,...,85931,21866,0,0,1회 이상,0,30회 이상,10회 이상,10회 이상,1회 이상
2,201807,TRAIN_000003,notAB,19226,20523,19062,6531,5933,40대,자녀성장(2),...,61518,16356,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,10회 이상
3,201807,TRAIN_000008,notAB,208224,199966,199862,62007,68078,30대,자녀출산기,...,62715,20512,0,0,10회 이상,559,1회 이상,1회 이상,1회 이상,1회 이상
4,201807,TRAIN_000010,notAB,100056,100037,100017,31378,18796,40대,자녀성장(1),...,30449,22512,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1077943,201812,TEST_99995,NaN,0,0,0,0,0,60대,노년생활,...,0,0,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
1077944,201812,TEST_99996,NaN,49990,50008,49025,17876,1231,30대,자녀출산기,...,1256,359,0,0,1회 이상,0,10회 이상,1회 이상,1회 이상,1회 이상
1077945,201812,TEST_99997,NaN,30004,30011,29996,13332,0,30대,자녀성장(1),...,0,0,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
1077946,201812,TEST_99998,NaN,37139,37999,42610,17362,63592,30대,가족구축기,...,48141,21273,0,0,1회 이상,240,40회 이상,1회 이상,1회 이상,1회 이상


In [5]:
# 결과 데이터는 제거한다.
all_df.drop('Group', axis=1, inplace=True)
all_df

,기준년월,ID,카드이용한도금액_B2M,카드이용한도금액_B1M,카드이용한도금액,CA한도금액,_1순위카드이용금액,연령,Life_Stage,거주시도명,...,청구금액_R3M,청구금액_B0,포인트_마일리지_환산_B0M,마일_적립포인트_R3M,할인건수_R3M,평잔_일시불_해외_6M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M
0,201807,TRAIN_000000,19723,20805,19354,7270,3681,40대,자녀성장(2),서울,...,46588,12226,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,10회 이상
1,201807,TRAIN_000002,77975,78730,88193,35207,24493,30대,자녀출산기,서울,...,85931,21866,0,0,1회 이상,0,30회 이상,10회 이상,10회 이상,1회 이상
2,201807,TRAIN_000003,19226,20523,19062,6531,5933,40대,자녀성장(2),부산,...,61518,16356,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,10회 이상
3,201807,TRAIN_000008,208224,199966,199862,62007,68078,30대,자녀출산기,서울,...,62715,20512,0,0,10회 이상,559,1회 이상,1회 이상,1회 이상,1회 이상
4,201807,TRAIN_000010,100056,100037,100017,31378,18796,40대,자녀성장(1),전북,...,30449,22512,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1077943,201812,TEST_99995,0,0,0,0,0,60대,노년생활,경기,...,0,0,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
1077944,201812,TEST_99996,49990,50008,49025,17876,1231,30대,자녀출산기,서울,...,1256,359,0,0,1회 이상,0,10회 이상,1회 이상,1회 이상,1회 이상
1077945,201812,TEST_99997,30004,30011,29996,13332,0,30대,자녀성장(1),경남,...,0,0,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
1077946,201812,TEST_99998,37139,37999,42610,17362,63592,30대,가족구축기,경남,...,48141,21273,0,0,1회 이상,240,40회 이상,1회 이상,1회 이상,1회 이상


In [6]:
# 결과 데이터는 제거한다.
all_df.drop(columns=['ID','기준년월'], axis=1, inplace=True)
all_df

,카드이용한도금액_B2M,카드이용한도금액_B1M,카드이용한도금액,CA한도금액,_1순위카드이용금액,연령,Life_Stage,거주시도명,연회비발생카드수_B0M,이용금액_할부_무이자_R12M,...,청구금액_R3M,청구금액_B0,포인트_마일리지_환산_B0M,마일_적립포인트_R3M,할인건수_R3M,평잔_일시불_해외_6M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M
0,19723,20805,19354,7270,3681,40대,자녀성장(2),서울,0개,5828,...,46588,12226,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,10회 이상
1,77975,78730,88193,35207,24493,30대,자녀출산기,서울,0개,10561,...,85931,21866,0,0,1회 이상,0,30회 이상,10회 이상,10회 이상,1회 이상
2,19226,20523,19062,6531,5933,40대,자녀성장(2),부산,0개,12098,...,61518,16356,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,10회 이상
3,208224,199966,199862,62007,68078,30대,자녀출산기,서울,1개이상,2204,...,62715,20512,0,0,10회 이상,559,1회 이상,1회 이상,1회 이상,1회 이상
4,100056,100037,100017,31378,18796,40대,자녀성장(1),전북,0개,25891,...,30449,22512,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1077943,0,0,0,0,0,60대,노년생활,경기,0개,0,...,0,0,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
1077944,49990,50008,49025,17876,1231,30대,자녀출산기,서울,0개,0,...,1256,359,0,0,1회 이상,0,10회 이상,1회 이상,1회 이상,1회 이상
1077945,30004,30011,29996,13332,0,30대,자녀성장(1),경남,0개,0,...,0,0,0,0,1회 이상,0,1회 이상,1회 이상,1회 이상,1회 이상
1077946,37139,37999,42610,17362,63592,30대,가족구축기,경남,0개,12509,...,48141,21273,0,0,1회 이상,240,40회 이상,1회 이상,1회 이상,1회 이상


In [7]:
all_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1077948 entries, 0 to 1077947
Data columns (total 39 columns):
 #   Column            Non-Null Count    Dtype 
---  ------            --------------    ----- 
 0   카드이용한도금액_B2M      1077948 non-null  int64 
 1   카드이용한도금액_B1M      1077948 non-null  int64 
 2   카드이용한도금액          1077948 non-null  int64 
 3   CA한도금액            1077948 non-null  int64 
 4   _1순위카드이용금액        1077948 non-null  int64 
 5   연령                1077948 non-null  object
 6   Life_Stage        1077948 non-null  object
 7   거주시도명             1077948 non-null  object
 8   연회비발생카드수_B0M      1077948 non-null  object
 9   이용금액_할부_무이자_R12M  1077948 non-null  int64 
 10  이용금액_일시불_R12M     1077948 non-null  int64 
 11  쇼핑_도소매_이용금액       1077948 non-null  int64 
 12  최대이용금액_일시불_R12M   1077948 non-null  int64 
 13  이용금액_할부_R12M      1077948 non-null  int64 
 14  할부금액_무이자_3M_R12M  1077948 non-null  int64 
 15  이용금액_오프라인_R6M     1077948 non-null  int64 
 16  할부금액_3M_R12M      

In [8]:
# LabelEncoder 학습
Encoder1 = LabelEncoder()
Encoder2 = LabelEncoder()
Encoder3 = LabelEncoder()
Encoder4 = LabelEncoder()
Encoder5 = LabelEncoder()
Encoder6 = LabelEncoder()
Encoder7 = LabelEncoder()
Encoder8 = LabelEncoder()
Encoder9 = LabelEncoder()

Encoder1.fit(all_df['연령'])
Encoder2.fit(all_df['Life_Stage'])
Encoder3.fit(all_df['거주시도명'])
Encoder4.fit(all_df['연회비발생카드수_B0M'])
Encoder5.fit(all_df['할인건수_R3M'])
Encoder6.fit(all_df['방문횟수_앱_R6M'])
Encoder7.fit(all_df['방문횟수_PC_R6M'])
Encoder8.fit(all_df['방문일수_PC_R6M'])
Encoder9.fit(all_df['인입횟수_ARS_R6M'])

LabelEncoder()

In [9]:
all_df['연령'] = Encoder1.transform(all_df['연령'])
all_df['Life_Stage'] = Encoder2.transform(all_df['Life_Stage'])
all_df['거주시도명'] = Encoder3.transform(all_df['거주시도명'])
all_df['연회비발생카드수_B0M'] = Encoder4.transform(all_df['연회비발생카드수_B0M'])
all_df['할인건수_R3M'] = Encoder5.transform(all_df['할인건수_R3M'])
all_df['방문횟수_앱_R6M'] = Encoder6.transform(all_df['방문횟수_앱_R6M'])
all_df['방문횟수_PC_R6M'] = Encoder7.transform(all_df['방문횟수_PC_R6M'])
all_df['방문일수_PC_R6M'] = Encoder8.transform(all_df['방문일수_PC_R6M'])
all_df['인입횟수_ARS_R6M'] = Encoder9.transform(all_df['인입횟수_ARS_R6M'])

In [10]:
# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_df)

,copy,True
,with_mean,True
,with_std,True


In [11]:
train_df['연령'] = Encoder1.transform(train_df['연령'])
train_df['Life_Stage'] = Encoder2.transform(train_df['Life_Stage'])
train_df['거주시도명'] = Encoder3.transform(train_df['거주시도명'])
train_df['연회비발생카드수_B0M'] = Encoder4.transform(train_df['연회비발생카드수_B0M'])
train_df['할인건수_R3M'] = Encoder5.transform(train_df['할인건수_R3M'])
train_df['방문횟수_앱_R6M'] = Encoder6.transform(train_df['방문횟수_앱_R6M'])
train_df['방문횟수_PC_R6M'] = Encoder7.transform(train_df['방문횟수_PC_R6M'])
train_df['방문일수_PC_R6M'] = Encoder8.transform(train_df['방문일수_PC_R6M'])
train_df['인입횟수_ARS_R6M'] = Encoder9.transform(train_df['인입횟수_ARS_R6M'])

In [13]:
# 입력과 결과로 나눈다.
X = train_df.drop(columns=['Group','ID','기준년월'])
y = train_df['Group']

In [14]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[-0.85032643, -0.82611875, -0.84573142, ..., -0.11592244,
         0.07677258, -4.52836119],
       [ 0.29852506,  0.31328535,  0.48971269, ..., -2.54843883,
        -3.07059378,  0.22083044],
       [-0.86012832, -0.83166578, -0.85139608, ..., -0.11592244,
         0.07677258, -4.52836119],
       ...,
       [-0.03361474, -0.0540188 , -0.17363478, ..., -0.11592244,
         0.07677258,  0.22083044],
       [ 0.30177921,  0.31853733,  0.41258016, ..., -0.11592244,
         0.07677258,  0.22083044],
       [ 0.01056275, -0.02927352, -0.20054192, ..., -0.11592244,
         0.07677258,  0.22083044]])

In [15]:
scaler_columns = X.columns.tolist()
scaler_columns

['카드이용한도금액_B2M',
 '카드이용한도금액_B1M',
 '카드이용한도금액',
 'CA한도금액',
 '_1순위카드이용금액',
 '연령',
 'Life_Stage',
 '거주시도명',
 '연회비발생카드수_B0M',
 '이용금액_할부_무이자_R12M',
 '이용금액_일시불_R12M',
 '쇼핑_도소매_이용금액',
 '최대이용금액_일시불_R12M',
 '이용금액_할부_R12M',
 '할부금액_무이자_3M_R12M',
 '이용금액_오프라인_R6M',
 '할부금액_3M_R12M',
 '이용금액_오프라인_B0M',
 '이용금액_오프라인_R3M',
 '_2순위쇼핑업종_이용금액',
 '_1순위업종_이용금액',
 '_3순위업종_이용금액',
 '정상청구원금_B5M',
 '정상입금원금_B5M',
 '정상입금원금_B2M',
 '정상입금원금_B0M',
 '정상청구원금_B2M',
 '정상청구원금_B0M',
 '청구금액_R6M',
 '청구금액_R3M',
 '청구금액_B0',
 '포인트_마일리지_환산_B0M',
 '마일_적립포인트_R3M',
 '할인건수_R3M',
 '평잔_일시불_해외_6M',
 '방문횟수_앱_R6M',
 '방문횟수_PC_R6M',
 '방문일수_PC_R6M',
 '인입횟수_ARS_R6M']

In [16]:
train_X = X2
train_y = y

In [17]:
le = LabelEncoder()
train_y = le.fit_transform(train_y)

### 기본 모델 사용하기
- 기본 모델 중에 만족하는 것을 찾았다면 하이퍼 파라미터 튜닝 과정은 생략하세요

In [18]:
model5 = LGBMClassifier(device='cpu', objective='multiclass', num_class=3, verbose=-1)

kfold = KFold(n_splits=5, shuffle=True, random_state=1)
r1 = cross_val_score(model5, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r1.mean()}')

f1_score_list.append(r1.mean())
model_name_list.append("LGBMClassifier")

평균 f1 Score : 0.9951584646699814


In [19]:
# CPU 기반 XGBoost 모델
model6 = XGBClassifier(
    n_jobs=-1,
    verbosity=0,
    use_label_encoder=False,
    eval_metric='mlogloss'
)

# 교차 검증
kfold = KFold(n_splits=10, shuffle=True, random_state=1)

# f1_weighted 사용
r2 = cross_val_score(model6, train_X, train_y, scoring='f1_micro', cv=kfold)
print(f'평균 f1 Score : {r2.mean():.4f}')

f1_score_list.append(r2.mean())
model_name_list.append("XGBClassifier")

평균 f1 Score : 0.9985


In [20]:
df = pd.DataFrame({
    'Model': model_name_list,
    'f1 score': f1_score_list
})
df = df.dropna()

In [21]:
df

,Model,f1 score
0,LGBMClassifier,0.995158
1,XGBClassifier,0.998460


In [22]:
final_model=model6.fit(train_X, train_y)

In [23]:
with open(best_model_path, 'wb') as fp:
    pickle.dump(model6, fp)
    pickle.dump(scalerX, fp)
    pickle.dump(scaler_columns, fp)
    pickle.dump(Encoder1, fp)
    pickle.dump(Encoder2, fp)
    pickle.dump(Encoder3, fp)
    pickle.dump(Encoder4, fp)
    pickle.dump(Encoder5, fp)
    pickle.dump(Encoder6, fp)
    pickle.dump(Encoder7, fp)
    pickle.dump(Encoder8, fp)
    pickle.dump(Encoder9, fp)
    pickle.dump(le, fp)  # ← LabelEncoder 객체 추가

print('저장완료')

저장완료
